In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter, defaultdict
import itertools
from tree_sitter_languages import get_parser
import ast
import re

# Load data
train_data = pd.read_csv("/home/info-sec-lab/Downloads/Aman/Codelite/train_original.csv")
test_data = pd.read_csv("/home/info-sec-lab/Downloads/Aman/Codelite/test_original.csv")

# Check column names
print("Train columns:", train_data.columns.tolist())
print("Test columns:", test_data.columns.tolist())

# Get unique languages
languages = sorted(train_data['language'].unique())
print(f"Number of languages: {len(languages)}")
print("Languages:", languages)

# Label encoding
label_encoder = LabelEncoder()
label_encoder.fit(languages)

train_data["label_id"] = label_encoder.transform(train_data["language"])
test_data["label_id"] = label_encoder.transform(test_data["language"])

print(f"\nTrain size: {len(train_data)}, Test size: {len(test_data)}")
print(f"Class distribution in train: {train_data['label_id'].value_counts().to_dict()}")

Train columns: ['Unnamed: 0', 'code', 'language']
Test columns: ['Unnamed: 0', 'code', 'language']
Number of languages: 21
Languages: ['bash', 'c', 'c#', 'c++', 'css', 'haskell', 'html', 'java', 'javascript', 'lua', 'markdown', 'objective-c', 'perl', 'php', 'python', 'r', 'ruby', 'scala', 'sql', 'swift', 'vb.net']

Train size: 152199, Test size: 47555
Class distribution in train: {1: 7681, 3: 7681, 2: 7681, 4: 7681, 5: 7681, 19: 7681, 6: 7681, 7: 7681, 12: 7681, 11: 7681, 16: 7681, 15: 7681, 14: 7681, 20: 7681, 18: 7681, 17: 7681, 0: 7680, 13: 7680, 8: 7679, 9: 5394, 10: 870}


In [ ]:
# Initialize parser
parser = get_parser("python")

def get_leaf_nodes(node):
    """Get all leaf nodes from AST."""
    if len(node.children) == 0:
        return [node]
    leaves = []
    for child in node.children:
        leaves.extend(get_leaf_nodes(child))
    return leaves

def get_path(node1, node2, max_length=10):
    """Get path between two nodes."""
    if node1 is None or node2 is None:
        return None
    
    # Get paths to root
    path1 = []
    n = node1
    while n:
        path1.append(n)
        n = n.parent
    
    path2 = []
    n = node2
    while n:
        path2.append(n)
        n = n.parent
    
    # Find lowest common ancestor
    lca = None
    for n in path1:
        if n in path2:
            lca = n
            break
    
    if lca is None:
        return None
    
    # Construct path
    path_to_lca = path1[:path1.index(lca)]
    path_from_lca = path2[:path2.index(lca)]
    
    path_str = []
    
    # Upward path
    for node in reversed(path_to_lca):
        path_str.append(f"↑{node.type}")
    
    # LCA
    path_str.append(lca.type)
    
    # Downward path
    for node in path_from_lca:
        path_str.append(f"↓{node.type}")
    
    if len(path_str) > max_length:
        return None
    
    return "|".join(path_str)

def extract_code_features(code, max_contexts=300, max_leaves=200):
    """Extract AST-based features from code."""
    try:
        tree = parser.parse(code.encode())
    except Exception as e:
        return []
    
    root = tree.root_node
    leaves = get_leaf_nodes(root)
    
    if len(leaves) > max_leaves:
        leaves = leaves[:max_leaves]
    
    contexts = []
    
    for i in range(0, len(leaves), 2):
        for j in range(i+1, min(i+5, len(leaves))):
            if len(contexts) >= max_contexts:
                break
            path = get_path(leaves[i], leaves[j])
            if path:
                # Use both node types and their text content
                start_text = leaves[i].text.decode('utf-8', errors='ignore')[:20]
                end_text = leaves[j].text.decode('utf-8', errors='ignore')[:20]
                contexts.append((leaves[i].type, path, leaves[j].type, start_text, end_text))
    
    return contexts

def extract_simple_features(code):
    """Extract simple lexical features as backup."""
    features = []
    
    # Common keywords across languages
    keywords = ['def', 'class', 'import', 'from', 'if', 'else', 'for', 'while', 
                'return', 'function', 'var', 'let', 'const', 'print', 'console.log',
                'public', 'private', 'protected', 'void', 'int', 'string', 'float',
                'def', 'end', 'begin', 'then', 'do', 'end', 'package', 'import']
    
    # Special characters
    special_chars = ['@', '#', '$', '&', '*', '::', '->', '=>', ':=', '++', '--']
    
    # Patterns
    patterns = [
        (r'def\s+\w+\(', 'py_func_def'),
        (r'function\s+\w+\(', 'js_func_def'),
        (r'public\s+class', 'java_class'),
        (r'#include', 'c_include'),
        (r'namespace', 'csharp_ns'),
        (r'module', 'ocaml_module'),
        (r'func\s+\w+\(', 'go_func'),
        (r'\$\w+', 'php_var'),
        (r'<-', 'r_assign'),
        (r'\(.*?\)\s*=>', 'arrow_func'),
    ]
    
    code_lower = code.lower()
    
    for kw in keywords:
        if kw in code_lower:
            features.append(f"kw_{kw}")
    
    for char in special_chars:
        if char in code:
            features.append(f"char_{char}")
    
    for pattern, name in patterns:
        if re.search(pattern, code):
            features.append(name)
    
    return features


In [3]:
# Build vocabularies
print("\nBuilding vocabularies...")

all_tokens = Counter()
all_paths = Counter()
all_simple_features = Counter()

for idx, row in train_data.iterrows():
    if idx % 100 == 0:
        print(f"Processing {idx}/{len(train_data)}...")
    
    code = row["code"]
    
    # AST-based features
    contexts = extract_code_features(code)
    for t1, path, t2, text1, text2 in contexts:
        all_tokens[t1] += 1
        all_tokens[t2] += 1
        all_paths[path] += 1
        if text1 and len(text1.strip()) > 0:
            all_tokens[f"TEXT_{text1[:10]}"] += 1
        if text2 and len(text2.strip()) > 0:
            all_tokens[f"TEXT_{text2[:10]}"] += 1
    
    # Simple features
    simple_feats = extract_simple_features(code)
    for feat in simple_feats:
        all_simple_features[feat] += 1



Building vocabularies...
Processing 0/152199...
Processing 100/152199...
Processing 200/152199...
Processing 300/152199...
Processing 400/152199...
Processing 500/152199...
Processing 600/152199...
Processing 700/152199...
Processing 800/152199...
Processing 900/152199...
Processing 1000/152199...
Processing 1100/152199...
Processing 1200/152199...
Processing 1300/152199...
Processing 1400/152199...
Processing 1500/152199...
Processing 1600/152199...
Processing 1700/152199...
Processing 1800/152199...
Processing 1900/152199...
Processing 2000/152199...
Processing 2100/152199...
Processing 2200/152199...
Processing 2300/152199...
Processing 2400/152199...
Processing 2500/152199...
Processing 2600/152199...
Processing 2700/152199...
Processing 2800/152199...
Processing 2900/152199...
Processing 3000/152199...
Processing 3100/152199...
Processing 3200/152199...
Processing 3300/152199...
Processing 3400/152199...
Processing 3500/152199...
Processing 3600/152199...
Processing 3700/152199..

In [4]:
# Build vocab with reasonable thresholds
def build_vocab(counter, min_freq=3, max_size=5000):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    sorted_items = sorted(counter.items(), key=lambda x: -x[1])
    
    for token, freq in sorted_items:
        if freq >= min_freq and len(vocab) < max_size:
            vocab[token] = len(vocab)
    
    return vocab

token2idx = build_vocab(all_tokens, min_freq=2, max_size=3000)
path2idx = build_vocab(all_paths, min_freq=2, max_size=2000)
simple2idx = build_vocab(all_simple_features, min_freq=1, max_size=500)

print(f"Token vocab size: {len(token2idx)}")
print(f"Path vocab size: {len(path2idx)}")
print(f"Simple features vocab size: {len(simple2idx)}")

def encode_sample(code, label_id, max_contexts=300):
    """Encode a code sample into feature vectors."""
    contexts = extract_code_features(code, max_contexts)
    simple_feats = extract_simple_features(code)
    
    encoded_contexts = []
    for t1, path, t2, text1, text2 in contexts:
        encoded_contexts.append((
            token2idx.get(t1, 1),
            path2idx.get(path, 1),
            token2idx.get(t2, 1)
        ))
    
    # Pad contexts
    while len(encoded_contexts) < max_contexts:
        encoded_contexts.append((0, 0, 0))
    encoded_contexts = encoded_contexts[:max_contexts]
    
    # Encode simple features as bag-of-words
    simple_vec = np.zeros(len(simple2idx))
    for feat in simple_feats:
        idx = simple2idx.get(feat, 1)
        if idx != 1:  # Not UNK
            simple_vec[idx] = 1
    
    return encoded_contexts, simple_vec, label_id


Token vocab size: 3000
Path vocab size: 2000
Simple features vocab size: 50


In [5]:
class Code2VecDataset(Dataset):
    def __init__(self, dataframe, max_contexts=300):
        self.df = dataframe
        self.max_contexts = max_contexts
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        contexts, simple_vec, label = encode_sample(
            row["code"], 
            row["label_id"],
            self.max_contexts
        )
        
        return {
            'contexts': torch.tensor(contexts, dtype=torch.long),
            'simple_features': torch.tensor(simple_vec, dtype=torch.float32),
            'label': torch.tensor(label, dtype=torch.long)
        }


In [6]:

class ImprovedCode2Vec(nn.Module):
    def __init__(self, token_vocab_size, path_vocab_size, simple_feat_size, num_classes, 
                 emb_dim=128, hidden_dim=256, dropout=0.3):
        super().__init__()
        
        # Embedding layers
        self.token_emb = nn.Embedding(token_vocab_size, emb_dim, padding_idx=0)
        self.path_emb = nn.Embedding(path_vocab_size, emb_dim, padding_idx=0)
        
        # Context processing
        self.context_lstm = nn.LSTM(
            input_size=emb_dim * 3,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        
        # Attention
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        
        # Simple features processing
        self.simple_processor = nn.Sequential(
            nn.Linear(simple_feat_size, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU()
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 64, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, contexts, simple_features):
        batch_size = contexts.shape[0]
        
        # Extract components
        t1 = contexts[..., 0]
        p = contexts[..., 1]
        t2 = contexts[..., 2]
        
        # Get embeddings
        e1 = self.token_emb(t1)
        ep = self.path_emb(p)
        e2 = self.token_emb(t2)
        
        # Combine context
        context_vec = torch.cat([e1, ep, e2], dim=-1)
        
        # Process through LSTM
        lstm_out, _ = self.context_lstm(context_vec)
        lstm_out = self.dropout(lstm_out)
        
        # Attention
        attn_weights = torch.softmax(self.attention(lstm_out).squeeze(-1), dim=1)
        context_attn = torch.sum(lstm_out * attn_weights.unsqueeze(-1), dim=1)
        
        # Process simple features
        simple_processed = self.simple_processor(simple_features)
        
        # Combine features
        combined = torch.cat([context_attn, simple_processed], dim=1)
        
        # Classify
        logits = self.classifier(combined)
        
        return logits


In [7]:
# Create datasets
train_dataset = Code2VecDataset(train_data, max_contexts=200)
test_dataset = Code2VecDataset(test_data, max_contexts=200)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

model = ImprovedCode2Vec(
    token_vocab_size=len(token2idx),
    path_vocab_size=len(path2idx),
    simple_feat_size=len(simple2idx),
    num_classes=len(languages)
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Training setup
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
criterion = nn.CrossEntropyLoss()

# Training loop
num_epochs = 15
best_accuracy = 0

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    for batch in train_loader:
        contexts = batch['contexts'].to(device)
        simple = batch['simple_features'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        outputs = model(contexts, simple)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_acc = 100. * correct / total
    scheduler.step()
    
    # Validation phase
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in test_loader:
            contexts = batch['contexts'].to(device)
            simple = batch['simple_features'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(contexts, simple)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    val_acc = 100. * val_correct / val_total
    
    print(f"Epoch {epoch+1}/{num_epochs}:")
    print(f"  Train Loss: {train_loss/len(train_loader):.4f}, Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_loss/len(test_loader):.4f}, Acc: {val_acc:.2f}%")
    
    if val_acc > best_accuracy:
        best_accuracy = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  ↳ New best model saved! Accuracy: {val_acc:.2f}%")

print(f"\nBest validation accuracy: {best_accuracy:.2f}%")

# Load best model and evaluate
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for batch in test_loader:
        contexts = batch['contexts'].to(device)
        simple = batch['simple_features'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(contexts, simple)
        probs = F.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

# Calculate metrics
accuracy = accuracy_score(all_labels, all_preds)
print(f"\nFinal Test Accuracy: {accuracy:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_preds,
    target_names=label_encoder.classes_,
    digits=4
))

# Per-class accuracy
print("\nPer-class Accuracy:")
class_acc = {}
for i, lang in enumerate(label_encoder.classes_):
    mask = np.array(all_labels) == i
    if mask.any():
        class_acc[lang] = accuracy_score(
            np.array(all_labels)[mask],
            np.array(all_preds)[mask]
        )
        print(f"{lang}: {class_acc[lang]:.4f}")

# Confusion matrix (optional)
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.close()

print("\nConfusion matrix saved as 'confusion_matrix.png'")


Using device: cuda
Model parameters: 2,254,294
Epoch 1/15:
  Train Loss: 1.1861, Acc: 65.35%
  Val Loss: 1.0012, Acc: 70.99%
  ↳ New best model saved! Accuracy: 70.99%
Epoch 2/15:
  Train Loss: 0.9684, Acc: 71.99%
  Val Loss: 0.9337, Acc: 73.06%
  ↳ New best model saved! Accuracy: 73.06%
Epoch 3/15:
  Train Loss: 0.8922, Acc: 74.04%
  Val Loss: 0.9201, Acc: 73.62%
  ↳ New best model saved! Accuracy: 73.62%
Epoch 4/15:
  Train Loss: 0.8273, Acc: 75.67%
  Val Loss: 0.9050, Acc: 74.10%
  ↳ New best model saved! Accuracy: 74.10%
Epoch 5/15:
  Train Loss: 0.7631, Acc: 77.32%
  Val Loss: 0.9127, Acc: 74.41%
  ↳ New best model saved! Accuracy: 74.41%
Epoch 6/15:
  Train Loss: 0.6938, Acc: 79.04%
  Val Loss: 0.9127, Acc: 74.49%
  ↳ New best model saved! Accuracy: 74.49%
Epoch 7/15:
  Train Loss: 0.6251, Acc: 80.91%
  Val Loss: 0.9395, Acc: 74.56%
  ↳ New best model saved! Accuracy: 74.56%
Epoch 8/15:
  Train Loss: 0.5571, Acc: 82.63%
  Val Loss: 0.9910, Acc: 74.63%
  ↳ New best model saved! A